# GARQ Metacell Learning Tutorial

In [1]:

import os
import torch
import random
import argparse
import matplotlib
import numpy as np
import scanpy as sc
import seaborn as sns
from model import GARQ
from matplotlib import rcParams
from alive_progress import alive_bar
from engine import train_one_epoch, warm_one_epoch, inference, init_gart_anchors
from data_utils import load_data, compute_metacell
from eval_utils import (
    plot_metacell_umap,
    plot_metacell_size,
    plot_celltype_purity,
    plot_compactness_separation,
)

/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. I

In [3]:
args = argparse.Namespace(
    data_file=["/home/zhangpeiru/data/RNA+ADT/GSE163120/GSE163120_rna.h5ad","/home/zhangpeiru/data/RNA+ADT/GSE163120/GSE163120_adt.h5ad"],
    data_type=["RNA","ADT"],
    save_name="GSE163120",
    type_key="celltype",
    n_GARQs=613,
    k_knn=5,
    epoch=300,
    batch_size=512,
    converge_threshold=10,
    seed=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.random.manual_seed(args.seed)
if args.device == "cuda":
    torch.cuda.manual_seed_all(args.seed)

device = torch.device(args.device)

In [4]:
adata_list, dataloader_train, dataloader_eval, input_dims = load_data(args)
omics_num = len(adata_list)

net = GARQ(
    input_dims=input_dims,
    data_types=args.data_type,
    entry_num=args.n_GARQs,
    k_knn=args.k_knn,
).to(device)

optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

print("n_GARQs:", args.n_GARQs)

init_gart_anchors(
    model=net,
    data_types=args.data_type,
    dataloader=dataloader_train,
    device=device,
)
if omics_num == 1:
    net.copy_decoder_q()

warm_epochs = min(50, int(args.epoch * 0.2))
print("warm_epochs:", warm_epochs)
print("======= Training Start =======")

with alive_bar(args.epoch, enrich_print=False) as bar:
    loss_rec_his = loss_vq_his = 1e7
    stable_epochs = 0

    for epoch in range(args.epoch):
        bar()
        if epoch < warm_epochs:
            warm_one_epoch(
                model=net,
                data_types=args.data_type,
                dataloader=dataloader_train,
                optimizer=optimizer,
                epoch=epoch,
                device=device,
            )
        else:
            loss_rec, loss_vq = train_one_epoch(
                model=net,
                data_types=args.data_type,
                dataloader=dataloader_train,
                optimizer=optimizer,
                epoch=epoch,
                device=device,
            )
            converge = (abs(loss_vq_his - loss_vq) <= 1e-5) and (
                abs(loss_rec_his - loss_rec) <= 1e-5
            )
            if converge:
                stable_epochs += 1
                if stable_epochs >= args.converge_threshold:
                    print("Early Stopping.")
                    break
            else:
                stable_epochs = 0
                loss_rec_his = loss_rec
                loss_vq_his = loss_vq

print("======= Training Done =======")

=======Loading and Preprocessing Data=======
Data of 2 omics in total


/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:220: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby('mean_bin')['dispersions']
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


/home/zhangpeiru/data/RNA+ADT/GSE163120/GSE163120_rna.h5ad loaded with shape [24559, 2200]
/home/zhangpeiru/data/RNA+ADT/GSE163120/GSE163120_adt.h5ad loaded with shape [24559, 170]
n_GARQs: 613


WARNING clustering 1024 points to 613 centroids: please provide at least 23907 training points


GARQ Initialization completed.
warm_epochs: 50
======= Training Start =======
Epoch 1, RNA: recon_loss=0.4297 , ADT: recon_loss=1.9261
Epoch 20, RNA: recon_loss=0.3523 , ADT: recon_loss=0.8212
Epoch 40, RNA: recon_loss=0.3457 , ADT: recon_loss=0.7804
Epoch 60, RNA: recon_loss=0.3410 recon_loss Q=0.3852 , ADT: recon_loss=0.7645 recon_loss Q=0.9577 , Anchers: loss=0.0116
Epoch 80, RNA: recon_loss=0.3368 recon_loss Q=0.3801 , ADT: recon_loss=0.7530 recon_loss Q=0.9361 , Anchers: loss=0.0116
Epoch 100, RNA: recon_loss=0.3335 recon_loss Q=0.3783 , ADT: recon_loss=0.7433 recon_loss Q=0.9312 , Anchers: loss=0.0110
Epoch 120, RNA: recon_loss=0.3312 recon_loss Q=0.3773 , ADT: recon_loss=0.7370 recon_loss Q=0.9279 , Anchers: loss=0.0104
Epoch 140, RNA: recon_loss=0.3286 recon_loss Q=0.3755 , ADT: recon_loss=0.7313 recon_loss Q=0.9244 , Anchers: loss=0.0097
Epoch 160, RNA: recon_loss=0.3271 recon_loss Q=0.3748 , ADT: recon_loss=0.7278 recon_loss Q=0.9236 , Anchers: loss=0.0096
Epoch 180, RNA: rec

In [6]:
print("======= Inference Start =======")
embeds, ids, delta_confs, rec_q_percent, loss_ancher = inference(
    model=net, data_types=args.data_type, data_loader=dataloader_eval, device=device
)

print("* Quantized Recon Percent:", rec_q_percent)
print("* Anchers Loss:", loss_ancher)

os.makedirs("./save/", exist_ok=True)

for i in range(omics_num):
    metacell_path = (
        "./save/" + args.save_name + "_" + args.data_type[i] + "_"
        + str(args.n_GARQs) + "metacell_k" + str(args.k_knn) + ".h5ad"
    )
    metacell_adata = compute_metacell(adata_list[i], ids, args)
    metacell_adata.write_h5ad(metacell_path)
    print(args.data_type[i] + " metacell data saved at:", metacell_path)

assignment_path = (
    "./save/" + args.save_name + "_" + str(args.n_GARQs) + "metacell_k" + str(args.k_knn) + "_ids.h5ad"
)
adata = sc.AnnData(embeds, dtype=np.float32)
adata.obs["metacell"] = ids
if args.type_key in adata_list[0].obs_keys():
    adata.obs[args.type_key] = adata_list[0].obs[args.type_key].values

import IPython.display
if not hasattr(IPython.display, "set_matplotlib_formats"):
    IPython.display.set_matplotlib_formats = lambda *args, **kwargs: None
    
sc.set_figure_params(figsize=(7, 7), dpi=300)
sc.pp.neighbors(adata, use_rep="X", metric="cosine")
sc.tl.umap(adata)
if args.type_key in adata.obs_keys():
    sc.pl.umap(
        adata,
        color=[args.type_key],
        save="_" + args.save_name + "_embedding_k" + str(args.k_knn) + ".png",
        palette=sns.color_palette("husl", np.unique(adata.obs[args.type_key].values).size),
        show=False,
    )
rcParams.update(matplotlib.rcParamsDefault)
adata.write_h5ad(assignment_path)
print("Metacell assignment saved at:", assignment_path)

fig_save_name = args.save_name + "_" + str(args.n_GARQs) + "metacell_k" + str(args.k_knn)
plot_metacell_umap(adata, fig_save_name)
plot_metacell_size(adata, fig_save_name)
if args.type_key in adata.obs_keys():
    plot_celltype_purity(adata, adata.obs[args.type_key], fig_save_name)
for i in range(omics_num):
    plot_compactness_separation(
        dataloader_train.dataset.raw_list[i].numpy(),
        adata,
        fig_save_name + "_" + args.data_type[i],
        omics_num > 1,
    )

print("======= Inference Done =======")

======= Inference Start =======
* Quantized Recon Percent: 0.8414806723594666
* Anchers Loss: 0.39044813001343703


/home/zhangpeiru/vscode/GARQ/data_utils.py:131: RuntimeWarning: Mean of empty slice.
  [data[meta_ids == i].mean(axis=0) for i in range(meta_ids.max() + 1)]
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/numpy/core/_methods.py:121: RuntimeWarning: divide by zero encountered in divide
  ret = um.true_divide(
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


RNA metacell data saved at: ./save/GSE163120_RNA_613metacell_k5.h5ad
ADT metacell data saved at: ./save/GSE163120_ADT_613metacell_k5.h5ad


/home/zhangpeiru/vscode/GARQ/data_utils.py:131: RuntimeWarning: Mean of empty slice.
  [data[meta_ids == i].mean(axis=0) for i in range(meta_ids.max() + 1)]
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/numpy/core/_methods.py:121: RuntimeWarning: divide by zero encountered in divide
  ret = um.true_divide(
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/zhangpeiru/.conda/envs/MetqQ2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


...

Computing Pairwise Correlation...
|████████████████████████████████████████| 25/25 [100%] in 2.1s (11.88/s) 
Correlation Saved to: ./save/GSE163120_RNA_24559cells_correlation.npy
* Average Compactness Score: 0.43208273535001174
* Average Separation Score: 0.4778294684643359


/home/zhangpeiru/vscode/GARQ/eval_utils.py:117: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat(


Computing Pairwise Correlation...
|████████████████████████████████████████| 25/25 [100%] in 1.4s (17.93/s) 
Correlation Saved to: ./save/GSE163120_ADT_24559cells_correlation.npy
* Average Compactness Score: 0.7142223788567837
* Average Separation Score: 0.17863167306441885


/home/zhangpeiru/vscode/GARQ/eval_utils.py:117: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat(


======= Inference Done =======
